# 03 回归分析与联动性动态研究

本 Notebook 完成两部分任务：

1. 以利率变动幅度为自变量，回归各市场事件日收益率
2. 计算全球市场滚动相关，并比较不同货币政策阶段的联动性变化

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

PROJECT_DIR = Path.cwd()
if PROJECT_DIR.name != 'T2_FED_world_stock_market':
    PROJECT_DIR = Path.cwd() / 'homework' / 'Topics_01' / 'T2_FED_world_stock_market' if (Path.cwd() / 'homework' / 'Topics_01' / 'T2_FED_world_stock_market').exists() else Path.cwd()
CODE_DIR = PROJECT_DIR / 'code'
if str(CODE_DIR) not in sys.path:
    sys.path.append(str(CODE_DIR))

from data_utils import MARKET_CONFIG
from event_study_utils import regression_on_event_day, rolling_correlation_matrix, average_corr_by_period

plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'Arial Unicode MS']
plt.rcParams['axes.unicode_minus'] = False
sns.set_style('whitegrid')

DATA_CLEAN = PROJECT_DIR / 'data_clean'
OUTPUT_DIR = PROJECT_DIR / 'output'

returns_df = pd.read_csv(DATA_CLEAN / 'global_index_returns.csv', index_col=0, parse_dates=True)
events_df = pd.read_csv(DATA_CLEAN / 'fed_events_aligned.csv', parse_dates=['date'])
for key in MARKET_CONFIG:
    col = f'{key}_event_date'
    events_df[col] = pd.to_datetime(events_df[col])

returns_df.head()

## 1. 事件日回归分析

In [ ]:
regression_results = []
regression_sample_dict = {}
for key in MARKET_CONFIG:
    summary, reg_df = regression_on_event_day(returns_df, events_df, key)
    regression_sample_dict[key] = reg_df
    if summary is not None:
        summary['market_label'] = MARKET_CONFIG[key]['label']
        regression_results.append(summary)

regression_df = pd.DataFrame(regression_results)
display(regression_df.sort_values('beta'))

In [ ]:
plot_df = regression_df.sort_values('beta').copy()
plt.figure(figsize=(12, 6))
colors = ['firebrick' if beta < 0 else 'steelblue' for beta in plot_df['beta']]
plt.bar(plot_df['market_label'], plot_df['beta'], color=colors)
plt.axhline(0, color='black', linewidth=1)
plt.title('各市场对美联储利率变动的回归敏感度')
plt.xlabel('市场')
plt.ylabel('Beta on change_bp')
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'regression_beta_bar.png', dpi=300, bbox_inches='tight')
plt.show()

## 2. 事件日散点图

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(16, 12), sharex=True)
axes = axes.flatten()
for idx, key in enumerate(MARKET_CONFIG):
    ax = axes[idx]
    reg_df = regression_sample_dict.get(key, pd.DataFrame())
    if reg_df.empty:
        ax.set_visible(False)
        continue
    sns.regplot(data=reg_df, x='change_bp', y='event_day_return', ax=ax, scatter_kws={'alpha': 0.7, 's': 35}, line_kws={'color': 'black'})
    ax.set_title(MARKET_CONFIG[key]['label'])
    ax.set_xlabel('利率变动幅度(bp)')
    ax.set_ylabel('事件日收益率')

plt.suptitle('事件日收益率与美联储利率变动幅度的关系', y=1.02, fontsize=15)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'event_day_regression_scatter.png', dpi=300, bbox_inches='tight')
plt.show()

## 3. 滚动相关与联动性动态

In [ ]:
rolling_corr_df = rolling_correlation_matrix(returns_df, window=60)
display(rolling_corr_df.tail())

In [ ]:
plt.figure(figsize=(15, 7))
selected_pairs = rolling_corr_df.columns[:6]
for col in selected_pairs:
    plt.plot(rolling_corr_df.index, rolling_corr_df[col], linewidth=1.2, label=col)
plt.title('主要市场对的 60 日滚动相关系数')
plt.xlabel('日期')
plt.ylabel('滚动相关系数')
plt.legend(ncol=2, fontsize=9)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'rolling_correlation_pairs.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
subperiods = {
    '宽松期_2009_2015': ('2009-01-01', '2015-12-31'),
    '收紧期_2015_2019': ('2015-01-01', '2019-12-31'),
    '极端宽松期_2020_2022': ('2020-01-01', '2022-12-31'),
}

period_corr_dict = {}
for name, (start_date, end_date) in subperiods.items():
    corr_mat = average_corr_by_period(returns_df, start_date, end_date)
    period_corr_dict[name] = corr_mat
    print(name)
    display(corr_mat)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for idx, (name, corr_mat) in enumerate(period_corr_dict.items()):
    labeled = corr_mat.copy()
    labeled.index = [MARKET_CONFIG[c]['label'] for c in labeled.index]
    labeled.columns = [MARKET_CONFIG[c]['label'] for c in labeled.columns]
    sns.heatmap(labeled, annot=True, fmt='.2f', cmap='RdYlBu_r', center=0, square=True, ax=axes[idx], cbar=idx == 2)
    axes[idx].set_title(name.replace('_', '\n'))

plt.suptitle('不同货币政策阶段的全球市场相关性热力图', y=1.05, fontsize=15)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'correlation_heatmaps_subperiods.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
period_summary = []
for name, corr_mat in period_corr_dict.items():
    vals = corr_mat.values
    upper = vals[np.triu_indices_from(vals, k=1)]
    period_summary.append({
        'period': name,
        'avg_corr': upper.mean(),
        'median_corr': np.median(upper),
        'max_corr': upper.max(),
        'min_corr': upper.min(),
    })
period_summary_df = pd.DataFrame(period_summary)
display(period_summary_df)

## 4. 保存结果与解释

In [ ]:
regression_df.to_csv(OUTPUT_DIR / 'regression_results.csv', index=False)
rolling_corr_df.to_csv(OUTPUT_DIR / 'rolling_correlations_60d.csv')
period_summary_df.to_csv(OUTPUT_DIR / 'subperiod_correlation_summary.csv', index=False)
for name, corr_mat in period_corr_dict.items():
    corr_mat.to_csv(OUTPUT_DIR / f'{name}_corr_matrix.csv')
print('回归与联动性结果已保存')

- 若某些市场的回归 beta 显著为负，说明美联储加息幅度越大，这些市场在事件日越容易出现负收益。
- 若宽松期与极端宽松期的平均相关性高于常规阶段，说明全球流动性环境会强化市场共振。
- 中国市场若回归显著性较弱或相关性变化相对平缓，可以从资本账户限制、汇率制度和政策独立性角度解释。